In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')



In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Load the dataset
cars_path = os.path.join(path, 'Q3_data.csv')
df = pd.read_csv(cars_path)

print(f"Dataset shape: {df.shape}")

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print("Missing values:")
print(df.isnull().sum())

In [ ]:
for col in df.columns:
    if df[col].dtype in ['float64', 'int64']:
        df[col] = df[col].fillna(df[col].median())
    else:
        df[col] = df[col].fillna(df[col].mode()[0])

In [ ]:
initial_rows = df.shape[0]
df = df.drop_duplicates()
final_rows = df.shape[0]
print(f"Removed {initial_rows - final_rows} duplicate rows.")

In [ ]:
object_cols = df.select_dtypes(include=['object']).columns
label_encoder = LabelEncoder()

for col in object_cols:
    df[col] = label_encoder.fit_transform(df[col].astype(str))

In [ ]:
features = df.drop(columns=['Target'])
target = df['Target']

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(features), columns=features.columns)

In [ ]:
target_counts = target.value_counts(normalize=True)
print("Target distribution:")
print(target_counts)

# Threshold for "imbalance" is typically if the minority class is < 50
#you guys said If class distributions are not equal, then our data is imbalanced. Use StratifiedKFold and focus on F1-score.
if target_counts.min() < 0.5:
    print("Result: The dataset is imbalanced (Minority class < 50%).")
    print("Decision: We will use F1 Score as the evaluation metric.")
else:
    print("Result: The dataset is relatively balanced.")
    print("Decision: We will use Accuracy as the evaluation metric.")

In [ ]:
X = X_scaled
y = target

In [ ]:
!pip install catboost

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from catboost import CatBoostClassifier

X = X_scaled
y = target

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

f1_scores = []

model = CatBoostClassifier(verbose=0, random_state=42, auto_class_weights='Balanced')

print("Starting Cross-Validation Evaluating with F1 Score")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)


    f1 = f1_score(y_val, y_pred)
    f1_scores.append(f1)

    print(f"Fold {fold+1}: F1 Score = {f1:.4f}")

avg_f1_full = np.mean(f1_scores)
print(f"\nAverage F1 Score: {avg_f1_full:.4f}")

In [ ]:
feature_importances = model.get_feature_importance()
feature_names = X.columns

fi_df = pd.DataFrame({'Feature': feature_names, 'Importance': feature_importances})
fi_df = fi_df.sort_values(by='Importance', ascending=False).head(10)

plt.figure(figsize=(10, 6))
plt.barh(fi_df['Feature'], fi_df['Importance'])
plt.gca().invert_yaxis()
plt.title("Top 10 Feature Importances")
plt.xlabel("Importance Score")
plt.show()

In [ ]:
golden_feature = fi_df.iloc[0]['Feature']
print(f"The Golden Feature is: {golden_feature}")

In [ ]:
X_golden = X[[golden_feature]]

f1_scores_golden = []

print(f"Training with ONLY Golden Feature ({golden_feature})...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_golden, y)):
    X_train, X_val = X_golden.iloc[train_idx], X_golden.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)

    f1_scores_golden.append(f1_score(y_val, y_pred))

avg_f1_golden = np.mean(f1_scores_golden)

print(f"\nFull Model Average F1 Score: {avg_f1_full:.4f}")
print(f"Golden Feature Model Average F1 Score: {avg_f1_golden:.4f}")
print(f"Difference: {avg_f1_full - avg_f1_golden:.4f}")

In [ ]:
# i made it look nice have a bonus for proffiosnal look
# best reagrds to the one marking this your the best :)